# DeltaCAT SQL Interface Examples

This notebook demonstrates how to use the SQL interface for DeltaCAT tables using DuckDB's zero-copy Arrow integration.

## Setup

First, let's import the necessary modules and initialize DeltaCAT:

In [ ]:
import deltacat as dc
import pyarrow as pa
import pandas as pd
from deltacat.sql import DeltaCATSQLGateway
from deltacat.catalog import create_table, write_to_table
from deltacat.storage import Schema, Field
from deltacat.types.media import ContentType

# Initialize DeltaCAT catalog
dc.init()

## Create Sample Tables

Let's create some sample tables to demonstrate SQL queries:

In [ ]:
# Create sample data for customers table
customers_data = pa.table({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'city': ['New York', 'Los Angeles', 'Chicago', 'New York', 'Chicago'],
    'age': [25, 30, 35, 28, 32]
})

# Create sample data for orders table
orders_data = pa.table({
    'order_id': [101, 102, 103, 104, 105, 106],
    'customer_id': [1, 2, 1, 3, 4, 2],
    'product': ['Laptop', 'Phone', 'Tablet', 'Laptop', 'Phone', 'Headphones'],
    'amount': [1200, 800, 500, 1200, 900, 150],
    'order_date': ['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-20']
})

# Define schemas
customers_schema = Schema.of([
    Field.of(0, 'customer_id', 'int64', False),
    Field.of(1, 'name', 'string', False),
    Field.of(2, 'city', 'string', False),
    Field.of(3, 'age', 'int64', False)
])

orders_schema = Schema.of([
    Field.of(0, 'order_id', 'int64', False),
    Field.of(1, 'customer_id', 'int64', False),
    Field.of(2, 'product', 'string', False),
    Field.of(3, 'amount', 'int64', False),
    Field.of(4, 'order_date', 'string', False)
])

In [ ]:
# Create tables in DeltaCAT catalog
create_table(
    name='customers',
    namespace='sales',
    schema=customers_schema,
    content_types=[ContentType.PARQUET]
)

create_table(
    name='orders',
    namespace='sales',
    schema=orders_schema,
    content_types=[ContentType.PARQUET]
)

# Write data to tables
write_to_table(
    data=customers_data,
    table='customers',
    namespace='sales',
    content_type=ContentType.PARQUET
)

write_to_table(
    data=orders_data,
    table='orders',
    namespace='sales',
    content_type=ContentType.PARQUET
)

print("Tables created successfully!")

## Initialize SQL Gateway

Create a SQL gateway to query DeltaCAT tables:

In [ ]:
# Create SQL gateway
sql = DeltaCATSQLGateway(auto_register_tables=True)

print(f"SQL Gateway initialized: {sql}")

## Basic SQL Queries

Now let's run some SQL queries on our DeltaCAT tables:

In [ ]:
# Simple SELECT query
result = sql.sql("SELECT * FROM sales.customers")
print("All customers:")
print(result.to_pandas())

In [ ]:
# WHERE clause
result = sql.sql("SELECT * FROM sales.orders WHERE amount > 800")
print("High-value orders:")
print(result.to_pandas())

In [ ]:
# Aggregation
result = sql.sql("""
    SELECT 
        product,
        COUNT(*) as order_count,
        SUM(amount) as total_revenue,
        AVG(amount) as avg_order_value
    FROM sales.orders
    GROUP BY product
    ORDER BY total_revenue DESC
""")
print("Product sales summary:")
print(result.to_pandas())

## Joins

DuckDB supports all standard SQL joins on DeltaCAT tables:

In [ ]:
# JOIN query
result = sql.sql("""
    SELECT 
        c.name,
        c.city,
        o.product,
        o.amount,
        o.order_date
    FROM sales.customers c
    JOIN sales.orders o ON c.customer_id = o.customer_id
    ORDER BY o.order_date
""")
print("Customer orders:")
print(result.to_pandas())

In [ ]:
# Complex aggregation with JOIN
result = sql.sql("""
    SELECT 
        c.city,
        COUNT(DISTINCT c.customer_id) as customer_count,
        COUNT(o.order_id) as order_count,
        SUM(o.amount) as total_revenue,
        AVG(o.amount) as avg_order_value
    FROM sales.customers c
    LEFT JOIN sales.orders o ON c.customer_id = o.customer_id
    GROUP BY c.city
    ORDER BY total_revenue DESC
""")
print("Revenue by city:")
print(result.to_pandas())

## Window Functions

DuckDB supports advanced SQL features like window functions:

In [ ]:
# Window functions
result = sql.sql("""
    SELECT 
        customer_id,
        order_id,
        product,
        amount,
        order_date,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) as order_seq,
        SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date) as running_total
    FROM sales.orders
    ORDER BY customer_id, order_date
""")
print("Order sequence and running totals:")
print(result.to_pandas())

## CTEs (Common Table Expressions)

Complex queries using CTEs:

In [ ]:
# CTE example
result = sql.sql("""
    WITH customer_metrics AS (
        SELECT 
            c.customer_id,
            c.name,
            c.city,
            COUNT(o.order_id) as order_count,
            COALESCE(SUM(o.amount), 0) as total_spent
        FROM sales.customers c
        LEFT JOIN sales.orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name, c.city
    )
    SELECT 
        name,
        city,
        order_count,
        total_spent,
        CASE 
            WHEN total_spent > 1000 THEN 'High Value'
            WHEN total_spent > 500 THEN 'Medium Value'
            ELSE 'Low Value'
        END as customer_segment
    FROM customer_metrics
    ORDER BY total_spent DESC
""")
print("Customer segmentation:")
print(result.to_pandas())

## Query Execution Plans

Examine query plans for optimization:

In [ ]:
# Get query execution plan
plan = sql.explain("""
    SELECT c.name, SUM(o.amount) as total
    FROM sales.customers c
    JOIN sales.orders o ON c.customer_id = o.customer_id
    WHERE c.city = 'New York'
    GROUP BY c.name
""")
print("Query execution plan:")
print(plan)

## DDL Operations

Create and drop tables via SQL interface:

In [ ]:
# Create a new table
new_schema = pa.schema([
    ('product_id', pa.int64()),
    ('product_name', pa.string()),
    ('category', pa.string()),
    ('price', pa.float64())
])

success = sql.create_table(
    'products',
    schema=new_schema,
    namespace='sales'
)
print(f"Table created: {success}")

# Write some data
products_data = pa.table({
    'product_id': [1, 2, 3],
    'product_name': ['Laptop', 'Phone', 'Tablet'],
    'category': ['Electronics', 'Electronics', 'Electronics'],
    'price': [1200.0, 800.0, 500.0]
})

write_to_table(
    data=products_data,
    table='products',
    namespace='sales',
    content_type=ContentType.PARQUET
)

# Query the new table
sql.refresh_tables()  # Refresh to pick up new table
result = sql.sql("SELECT * FROM sales.products")
print("\nProducts table:")
print(result.to_pandas())

## Performance: Zero-Copy Arrow Integration

The key advantage of this integration is zero-copy access to Arrow data:

In [ ]:
# Large aggregation query - runs directly on Arrow data
import time

start = time.time()
result = sql.sql("""
    SELECT 
        COUNT(*) as total_orders,
        COUNT(DISTINCT customer_id) as unique_customers,
        SUM(amount) as total_revenue,
        AVG(amount) as avg_order_value,
        MIN(amount) as min_order,
        MAX(amount) as max_order
    FROM sales.orders
""")
elapsed = time.time() - start

print(f"Query executed in {elapsed:.3f} seconds")
print("\nResults:")
print(result.to_pandas())

# The query runs directly on Arrow data without serialization/deserialization
print("\n✅ Zero-copy: DuckDB queries Arrow Datasets directly!")

## Cleanup

In [ ]:
# Close SQL gateway
sql.close()
print("SQL Gateway closed")

## Summary

This notebook demonstrated:

1. **Zero-copy SQL access** to DeltaCAT tables via DuckDB
2. **Full SQL support** including joins, aggregations, window functions, and CTEs
3. **DDL operations** to create and manage tables
4. **Query optimization** with execution plans
5. **High performance** through Arrow's columnar format

The integration leverages:
- DeltaCAT for data lake management
- Arrow for efficient columnar storage
- DuckDB for world-class SQL execution
- Zero-copy data access for optimal performance